# Противоречия в человеческой разметке: размер

`matches.parquet` — **человеческая** разметка (в удалённом старом пайплайне аргумент
назывался `--matches_human_path`, таргет строго бинарный).
`matches_llm.parquet` — разметка LLM, таргет дробный: доли голосов из 9 прогонов.

Вопрос: последовательно ли размечено правило «разный размер → разные товары».

In [1]:
import polars as pl
import json
import re

items = pl.read_parquet('data/items_human.parquet')
matches = pl.read_parquet('data/matches.parquet')

print('matches.parquet — таргет:', sorted(matches['target'].unique().to_list()))
llm_head = pl.read_parquet('data/matches_llm.parquet').head(200_000)
print('matches_llm.parquet — таргет:', sorted(llm_head['target'].unique().to_list())[:6], '...')
print('\nДробные значения = доли голосов LLM (8/9 = 0.888..., 2/9 = 0.222...)')

matches.parquet — таргет: [0.0, 1.0]
matches_llm.parquet — таргет: [0.0, 0.1111111111111111, 0.2222222222222222, 0.3333333333333333, 0.4444444444444444, 0.5555555555555556] ...

Дробные значения = доли голосов LLM (8/9 = 0.888..., 2/9 = 0.222...)


## 1. Извлекаем размер, цвет, бренд

In [2]:
SIZE_KEYS = ('размер', 'российский размер', 'размер производителя')
COLOR_KEYS = ('цвет товара', 'цвет', 'название цвета')
BRAND_KEYS = ('бренд', 'бренд в одежде и обуви')


def first(attrs, keys):
    for k in keys:
        v = attrs.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip().lower()
    return None


category = dict(zip(items['id'].to_list(), items['category'].to_list()))
raw = {}
for item_id, name, attrs in zip(items['id'].to_list(), items['name'].to_list(), items['attributes'].to_list()):
    d = json.loads(attrs)
    raw[item_id] = (name, first(d, SIZE_KEYS), first(d, COLOR_KEYS), first(d, BRAND_KEYS))

print('товаров:', len(raw))

товаров: 711304


## 2. Масштаб противоречий

Берём пары, где размер указан у обоих товаров и различается.
Если разметка последовательна, одна из колонок должна быть близка к нулю.

In [3]:
rows = []
for a, b, t in zip(matches['id1'], matches['id2'], matches['target']):
    na, sa, ca, ba = raw[a]
    nb, sb, cb, bb = raw[b]
    if sa and sb and sa != sb:
        rows.append({'cat': category[a], 'target': t, 'id1': a, 'id2': b,
                     'name1': na, 'name2': nb, 'size1': sa, 'size2': sb,
                     'color1': ca, 'color2': cb})

diff = pl.DataFrame(rows)
summary = (diff.group_by('cat')
           .agg(pl.len().alias('пар'),
                (pl.col('target') == 1).sum().alias('помечено_один_товар'))
           .with_columns((pl.col('помечено_один_товар') / pl.col('пар')).alias('доля_противоречий'))
           .sort('пар', descending=True))
summary.head(10)

cat,пар,помечено_один_товар,доля_противоречий
str,u32,u32,f64
"""Одежда""",19036,2260,0.118722
"""Обувь""",11939,1221,0.10227
"""Спорт и отдых""",232,44,0.189655
"""Аптека""",113,26,0.230088
"""Строительство и ремонт""",10,4,0.4
"""Галантерея и аксессуары""",8,0,0.0
"""Автотовары""",8,1,0.125


In [4]:
for c in ('Одежда', 'Обувь'):
    sub = diff.filter(pl.col('cat') == c)
    pos = (sub['target'] == 1).sum()
    print(f'{c}: {len(sub)} пар с разным размером | '
          f'«разные» {len(sub)-pos} ({(len(sub)-pos)/len(sub):.1%}) | '
          f'«один товар» {pos} ({pos/len(sub):.1%})')

Одежда: 19036 пар с разным размером | «разные» 16776 (88.1%) | «один товар» 2260 (11.9%)
Обувь: 11939 пар с разным размером | «разные» 10718 (89.8%) | «один товар» 1221 (10.2%)


## 3. Настоящие ли это противоречия?

Разделяем случаи «один товар при разном размере» на два типа:
- **числа размеров пересекаются** — скорее разный формат записи (`45` и `45 46 eu`)
- **не пересекаются** — размеры реально разные, это противоречие разметки

In [5]:
def nums(s):
    return set(re.findall(r'\d+', s))


contra = diff.filter((pl.col('target') == 1) & pl.col('cat').is_in(['Одежда', 'Обувь']))
fmt, real = [], []
for r in contra.iter_rows(named=True):
    (fmt if nums(r['size1']) & nums(r['size2']) else real).append(r)

print(f'помечено «один товар» при разном размере: {contra.height}')
print(f'  формат записи (числа пересекаются): {len(fmt)} ({len(fmt)/contra.height:.0%})')
print(f'  размеры реально разные:             {len(real)} ({len(real)/contra.height:.0%})')

помечено «один товар» при разном размере: 3481
  формат записи (числа пересекаются): 392 (11%)
  размеры реально разные:             3089 (89%)


### Кейс 1 — разный формат записи, метка «один товар» верна

In [6]:
for r in fmt[:6]:
    print(f"A: {r['name1'][:58]:<58} размер={r['size1']:<14} цвет={r['color1']}")
    print(f"B: {r['name2'][:58]:<58} размер={r['size2']:<14} цвет={r['color2']}")
    print('-' * 100)

A: кроссовки yeezy                                            размер=45             цвет=шоколадный
B: кроссовки унисекс adidas 350-2602-56 коричневые 46 eu      размер=45 46 eu       цвет=коричневый
----------------------------------------------------------------------------------------------------
A: костюм хирурга универсальный голубой                       размер=48             цвет=голубой
B: sirius plus / костюм медицинский                           размер=48-182         цвет=голубой
----------------------------------------------------------------------------------------------------
A: кроссовки new balance                                      размер=eu 45,5        цвет=кремовый
B: кроссовки мужские new balance high-top trainers бежевые 46 размер=45.5 46.5 eu   цвет=бежевый
----------------------------------------------------------------------------------------------------
A: бюстгальтер plus size с сшивными чашками и кружевом        размер=90i            цвет=синий
B: бюстгальте

### Кейс 2 — размеры реально разные, метка «один товар» противоречит правилу

In [7]:
for r in real[:6]:
    print(f"A: {r['name1'][:58]:<58} размер={r['size1']:<14} цвет={r['color1']}")
    print(f"B: {r['name2'][:58]:<58} размер={r['size2']:<14} цвет={r['color2']}")
    print('-' * 100)

A: брюки спортивные acoola                                    размер=170            цвет=хаки
B: брюки acoola                                               размер=116            цвет=зеленый
----------------------------------------------------------------------------------------------------
A: кроссовки nike jordan                                      размер=36             цвет=белый
B: кеды унисекс nike 1502-20242 бежевые 38 eu                 размер=37 38 eu       цвет=jordan 1 low "fragment x scott"
----------------------------------------------------------------------------------------------------
A: блузка acoola                                              размер=110            цвет=черный
B: acoola футболка acoola, размер 140, черный                 размер=140            цвет=черный
----------------------------------------------------------------------------------------------------
A: кроссовки dunks nike                                       размер=36             цвет=разноцвет

### Кейс 3 — доминирующее правило, но шум есть и здесь

In [8]:
neg = diff.filter((pl.col('target') == 0) & pl.col('cat').is_in(['Одежда', 'Обувь']))
for r in neg.head(6).iter_rows(named=True):
    print(f"A: {r['name1'][:58]:<58} размер={r['size1']:<14} цвет={r['color1']}")
    print(f"B: {r['name2'][:58]:<58} размер={r['size2']:<14} цвет={r['color2']}")
    print('-' * 100)

# пары вида 50-52 / 52 — почти наверняка один размер, но помечены «разные»
print('\nпомечено «разные», хотя размеры пересекаются по числам:')
n = sum(1 for r in neg.iter_rows(named=True) if nums(r['size1']) & nums(r['size2']))
print(f'  {n} пар ({n/neg.height:.1%} от негативов)')

A: сандалии fre gamo                                          размер=43             цвет=черный
B: fre gamo / сандалии                                        размер=41             цвет=желтый
----------------------------------------------------------------------------------------------------
A: шлепанцы under armour ua u ansa elevate sl                 размер=us 10          цвет=черный
B: under armour шлепанцы under armour, размер 9 us, черный    размер=9 us           цвет=черный
----------------------------------------------------------------------------------------------------
A: kapika дутики:чёрный:36                                    размер=36             цвет=None
B: дутики kapika                                              размер=35             цвет=черный
----------------------------------------------------------------------------------------------------
A: кеды женские puma mayze stack luxe wns белые 37 eu         размер=36 37 eu       цвет=белый
B: кроссовки adidas smooth r

## 4. Сравнение с LLM-разметкой

Берём только уверенные метки LLM (ровно 0.0 или 1.0), чтобы сравнивать с бинарной
человеческой. Это смещает сравнение в пользу LLM — там, где она сомневалась,
противоречия как раз и были бы.

In [9]:
llm = pl.read_parquet('data/matches_llm.parquet')
conf = llm.filter((pl.col('target') == 0.0) | (pl.col('target') == 1.0)).sample(400_000, seed=0)
need = pl.concat([conf['id1'], conf['id2']]).unique()

sub = (pl.scan_parquet('data/items.parquet')
       .filter(pl.col('id').is_in(need.implode()))
       .select('id', 'attributes', 'category')
       .collect(engine='streaming'))

llm_size, llm_cat = {}, {}
for i, a, c in zip(sub['id'].to_list(), sub['attributes'].to_list(), sub['category'].to_list()):
    llm_size[i] = first(json.loads(a), SIZE_KEYS)
    llm_cat[i] = c

print('подгружено товаров:', sub.height)

подгружено товаров: 735305


In [10]:
print(f'{"категория":<12} {"человек":>22} {"LLM":>22}')
print('-' * 58)
for c in ('Одежда', 'Обувь'):
    h = diff.filter(pl.col('cat') == c)
    h_rate = (h['target'] == 1).sum() / h.height

    l = [t for a, b, t in zip(conf['id1'], conf['id2'], conf['target'])
         if llm_cat.get(a) == c and llm_size.get(a) and llm_size.get(b)
         and llm_size[a] != llm_size[b]]
    l_rate = sum(1 for t in l if t == 1.0) / len(l) if l else float('nan')

    print(f'{c:<12} {h_rate:>21.1%} {l_rate:>21.1%}')

print('\nдоля пар, где разный размер помечен как «один товар»')
print('чем ниже, тем последовательнее разметка')

категория                   человек                    LLM
----------------------------------------------------------


Одежда                       11.9%                  4.7%


Обувь                        10.2%                  2.4%

доля пар, где разный размер помечен как «один товар»
чем ниже, тем последовательнее разметка


## 5. Что дала бы нормализация размеров

Проверяем, сколько противоречий исчезло бы, если привести запись размеров к канону:
запятая → точка, срезание роста (`48-182` → `48`), разбор перечислений (`45 46 eu`).

In [11]:
def norm_size(s):
    s = s.replace(',', '.')
    s = re.sub(r'\b(eu|ru|us|uk)\b', '', s)
    return set(re.findall(r'\d+(?:\.\d+)?', s))


fixed = sum(1 for r in contra.iter_rows(named=True)
            if norm_size(r['size1']) & norm_size(r['size2']))

print(f'противоречий всего:                 {contra.height}')
print(f'исчезло бы после нормализации:      {fixed} ({fixed/contra.height:.0%})')
print(f'осталось бы неустранимых:           {contra.height - fixed}')
print(f'\nв масштабе всех пар: {fixed} из {matches.height} = {fixed/matches.height:.3%}')

противоречий всего:                 3481
исчезло бы после нормализации:      304 (9%)
осталось бы неустранимых:           3177

в масштабе всех пар: 304 из 365654 = 0.083%
